# 02 — Spatial Traffic Analysis

Exploratory analysis of the METR-LA traffic time-series.

- Temporal patterns (rush hours, weekday vs weekend)
- Spatial correlation between neighbouring sensors
- Missing data audit
- Feature distributions before / after Z-score normalisation

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import gridspec

from config import DATA_DIR, cfg
from src.dataset import load_traffic_h5, add_time_features
from src.utils import StandardScaler

sns.set_theme(style='darkgrid', palette='muted')
print('Setup done')

## 1. Load raw traffic data

In [ ]:
df = load_traffic_h5(DATA_DIR / 'metr-la' / 'metr-la.h5')
print(df.shape)  # (T, N)
df.head(3)

## 2. Missing data audit

In [ ]:
zero_frac = (df == 0).mean()
nan_frac  = df.isna().mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
zero_frac.hist(bins=40, ax=axes[0], color='coral')
axes[0].set_title('Zero-value fraction per sensor')
axes[0].set_xlabel('Fraction of zeros')

nan_frac.hist(bins=40,  ax=axes[1], color='steelblue')
axes[1].set_title('NaN fraction per sensor')
axes[1].set_xlabel('Fraction of NaNs')

plt.tight_layout()
plt.show()

print(f'Sensors with >5% zeros: {(zero_frac > 0.05).sum()}')
print(f'Global NaN rate:         {nan_frac.mean():.4%}')

## 3. Daily traffic patterns

In [ ]:
mean_flow = df.mean(axis=1).to_frame('flow')
mean_flow['hour'] = mean_flow.index.hour
mean_flow['dow']  = mean_flow.index.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly average
hourly = mean_flow.groupby('hour')['flow'].mean()
hourly.plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Average traffic flow by hour-of-day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean flow (mph)')
axes[0].axvspan(7, 9,   alpha=0.1, color='red',   label='AM peak')
axes[0].axvspan(17, 19, alpha=0.1, color='orange', label='PM peak')
axes[0].legend()

# Day-of-week distribution
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
sns.boxplot(data=mean_flow, x='dow', y='flow', order=order, ax=axes[1], palette='Set2')
axes[1].set_title('Traffic flow distribution by day-of-week')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Spatial correlation (first sensor vs neighbours)

In [ ]:
import pickle

with open(DATA_DIR / 'metr-la' / 'adj_mx.pkl', 'rb') as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')

# Compute correlation between sensor 0 and all others
corr = df.corr().values[0]  # correlation with sensor 0

plt.figure(figsize=(10, 4))
plt.scatter(range(len(corr)), corr, c=corr, cmap='RdYlGn', s=15, alpha=0.7)
plt.colorbar(label='Pearson r')
plt.xlabel('Sensor index')
plt.ylabel('Correlation with sensor 0')
plt.title('Spatial Correlation — Sensor 0 vs All Others')
plt.tight_layout()
plt.show()

print(f'Mean absolute correlation: {np.abs(corr).mean():.3f}')

## 5. Feature engineering & normalisation

In [ ]:
arr       = add_time_features(df)        # (T, N, 5)
T_total   = arr.shape[0]
n_train   = int(T_total * cfg.data.train_ratio)

scaler = StandardScaler().fit(arr[:n_train])
arr_n  = scaler.transform(arr)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(arr[:, :, 0].flatten(),   bins=60, color='coral',     alpha=0.8, label='raw')
axes[0].set_title('Raw flow values')
axes[1].hist(arr_n[:, :, 0].flatten(), bins=60, color='steelblue', alpha=0.8, label='normalised')
axes[1].set_title('Normalised flow values (Z-score)')

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()